# GeneWeaver — GPU verification (Colab)

Runs the `@cuda.jit` alignment kernel on real NVIDIA hardware and produces the
CPU-vs-GPU numbers for the mid-project review.

**Before running: Runtime -> Change runtime type -> Hardware accelerator: T4 GPU.**

The notebook does four things:

1. proves the CUDA kernel compiles and returns the same answer as the CPU path
2. benchmarks the pure-Python baseline against CUDA on identical data
3. scans a real human chromosome (chr21, ~46.7 Mbp) end to end
4. writes `gpu_results.md` for you to paste into the review


## 1. Confirm the GPU

In [ ]:

!nvidia-smi

import subprocess

if subprocess.run(["which", "nvidia-smi"], capture_output=True).returncode != 0:
    raise SystemExit(
        "No GPU in this runtime. Runtime -> Change runtime type -> T4 GPU, then rerun."
    )

## 2. Install dependencies and clone the repo

In [ ]:

REPO = "https://github.com/Kushal-191105/GeneWeaver.git"
BRANCH = "raji"

!pip install -q biopython
# Numba's CUDA target ships separately in newer versions; install both and let
# pip settle it. Harmless if one is already present.
!pip install -q numba numba-cuda 2>/dev/null || pip install -q numba

import os

if not os.path.isdir("GeneWeaver"):
    !git clone --branch $BRANCH --depth 1 $REPO
else:
    !cd GeneWeaver && git pull

os.chdir("/content/GeneWeaver")

import sys

if "/content/GeneWeaver" not in sys.path:
    sys.path.insert(0, "/content/GeneWeaver")

print("\nrepo contents:")
!ls -la
print("\ndata:")
!ls -lh data

## 3. Does Numba see the GPU?

In [ ]:

import numba
from numba import cuda

print("numba:", numba.__version__)
print("cuda.is_available():", cuda.is_available())

if not cuda.is_available():
    print("\nDiagnostics:")
    try:
        cuda.detect()
    except Exception as error:
        print("cuda.detect() failed:", error)
    raise SystemExit("Numba cannot reach the GPU - fix this before continuing.")

device = cuda.get_current_device()
free, total = cuda.current_context().get_memory_info()

print("device:", device.name.decode() if isinstance(device.name, bytes) else device.name)
print("SM (multiprocessor) count:", device.MULTIPROCESSOR_COUNT)
print("compute capability:", device.compute_capability)
print(f"VRAM: {free / 1024 ** 3:.2f} GiB free of {total / 1024 ** 3:.2f} GiB")

## 4. The kernel's first real execution

This is the cell that closes the Week 2 gap: `src/cuda_kernels.alignment_kernel`
JIT-compiles for the device, runs, and its output is compared base-for-base
against the CPU implementation.

In [ ]:

import numpy as np

from src.chunking import encode_bases
from src.gpu_alignment import count_mismatches_cuda, count_mismatches_numpy
from src.parser import load_sequence_dataset, read_targets

dataset = load_sequence_dataset("data/genome.fasta", limit=50)
target = read_targets("data/targets.csv")[0]

sequence = "".join(dataset["sequence"].tolist())
sequence_array = encode_bases(sequence)
target_array = encode_bases(target)

print("sequence bases:", f"{sequence_array.size:,}")
print("target:", target, f"({target_array.size} bases)")

cuda_counts = count_mismatches_cuda(sequence_array, target_array)
numpy_counts = count_mismatches_numpy(sequence_array, target_array)

print("\nkernel compiled and ran:", cuda_counts.size, "windows scored")
print("CUDA == CPU on every window:",
      np.array_equal(cuda_counts.astype(np.int32), numpy_counts.astype(np.int32)))
print("perfect matches found by kernel:", int((cuda_counts == 0).sum()))
print("first 12 mismatch counts:", cuda_counts[:12].tolist())

assert np.array_equal(cuda_counts.astype(np.int32), numpy_counts.astype(np.int32)), \
    "CUDA and CPU disagree - do not present this until it is fixed"

## 5. CPU baseline vs CUDA on identical data

In [ ]:

import time

import pandas as pd

from src.pipeline import run_chunked_alignment

SLICE_BASES = 2_000_000

big = load_sequence_dataset("data/genome.fasta")
sequence = "".join(big["sequence"].tolist())[:SLICE_BASES]

slice_frame = pd.DataFrame({
    "sequence_id": ["benchmark_slice"],
    "sequence": [sequence],
    "length": [len(sequence)],
})

print(f"workload: {len(sequence):,} bases x 1 target\n")

cpu = run_chunked_alignment(slice_frame, target, mode="cpu", max_mismatches=2)
print(f"pure-Python baseline : {cpu['elapsed']:.3f} s "
      f"({len(sequence) / cpu['elapsed'] / 1e3:,.0f} kbp/s)")

gpu = run_chunked_alignment(slice_frame, target, mode="gpu", max_mismatches=2)
print(f"CUDA ({gpu['backend']})        : {gpu['elapsed']:.3f} s "
      f"({len(sequence) / gpu['elapsed'] / 1e6:,.1f} Mbp/s)")

speedup = cpu["elapsed"] / gpu["elapsed"]
agree = (sorted(m["position"] for m in cpu["matches"])
         == sorted(m["position"] for m in gpu["matches"]))

print(f"\nGPU speedup: {speedup:.1f}x")
print("same matches from both backends:", agree)
print("matches found:", len(gpu["matches"]))

### Run the project's own benchmark script

The same comparison through `benchmark.py`, so the review sees the CLI output
rather than notebook cells.

In [ ]:

!python benchmark.py --limit 200

## 6. A real human chromosome, end to end

In [ ]:

!bash scripts/download_chromosome.sh 21
!ls -lh data/*.fa.gz

In [ ]:

CHROMOSOME = "data/Homo_sapiens.GRCh38.dna.chromosome.21.fa.gz"

free_before, total_vram = cuda.current_context().get_memory_info()

!python main.py --input $CHROMOSOME --limit 0 --mode gpu --output results/chr21_matches.csv

free_after, _ = cuda.current_context().get_memory_info()

print(f"\nVRAM used during the scan: "
      f"{(free_before - free_after) / 1024 ** 2:.0f} MiB of "
      f"{total_vram / 1024 ** 2:.0f} MiB")
print("(chunk size caps VRAM use, so genome size does not)")

In [ ]:

# Chromosome throughput, measured in-process so it can be quoted precisely.
chromosome = load_sequence_dataset(CHROMOSOME)
bases = int(chromosome["length"].sum())

result = run_chunked_alignment(chromosome, target, mode="gpu", max_mismatches=2)

chromosome_rate = bases / result["elapsed"]
baseline_rate = len(sequence) / cpu["elapsed"]

print(f"chromosome 21: {bases:,} bases in {result['chunks']} chunks")
print(f"CUDA scan    : {result['elapsed']:.2f} s "
      f"({chromosome_rate / 1e6:,.1f} Mbp/s), {len(result['matches'])} matches")
print(f"\nsame work on the pure-Python baseline would take "
      f"{bases / baseline_rate / 60:.1f} min (extrapolated)")
print(f"whole genome (3.1 Gbp) on CUDA: "
      f"{3.1e9 / chromosome_rate / 60:.1f} min (extrapolated)")

## 7. Results block for the review

In [ ]:

lines = [
    "# GeneWeaver - GPU verification results",
    "",
    f"- device: {device.name.decode() if isinstance(device.name, bytes) else device.name}"
    f" ({device.MULTIPROCESSOR_COUNT} SMs, {total_vram / 1024 ** 3:.1f} GiB VRAM)",
    f"- numba: {numba.__version__}",
    "",
    "## Kernel correctness",
    f"- windows scored by the CUDA kernel: {cuda_counts.size:,}",
    "- CUDA output identical to the CPU implementation: yes",
    "",
    f"## CPU vs GPU ({len(sequence):,} bases, 1 target)",
    f"- pure-Python baseline: {cpu['elapsed']:.3f} s ({baseline_rate / 1e3:,.0f} kbp/s)",
    f"- CUDA: {gpu['elapsed']:.3f} s ({len(sequence) / gpu['elapsed'] / 1e6:,.1f} Mbp/s)",
    f"- speedup: {speedup:.1f}x",
    "- both backends returned the same matches: "
    f"{'yes' if agree else 'NO - investigate'}",
    "",
    "## Chromosome 21 (real GRCh38 data)",
    f"- bases: {bases:,} in {result['chunks']} chunks of {result['chunk_size']:,}",
    f"- CUDA scan: {result['elapsed']:.2f} s ({chromosome_rate / 1e6:,.1f} Mbp/s)",
    f"- matches: {len(result['matches'])}",
    f"- VRAM used: {(free_before - free_after) / 1024 ** 2:.0f} MiB "
    f"(set by chunk size, not genome size)",
    "",
    "## Extrapolations",
    f"- chr21 on the pure-Python baseline: {bases / baseline_rate / 60:.1f} min",
    f"- whole genome (3.1 Gbp) on CUDA: {3.1e9 / chromosome_rate / 60:.1f} min",
]

report = "\n".join(lines)
print(report)

with open("gpu_results.md", "w") as handle:
    handle.write(report + "\n")

try:
    from google.colab import files

    files.download("gpu_results.md")
except Exception:
    print("\n(download gpu_results.md from the file browser on the left)")

## What to take to the review

- a screenshot of **cell 3** (device name, SMs, VRAM) — proof it ran on real hardware
- a screenshot of **cell 4** — the kernel's output matching the CPU exactly
- the **speedup number** from cell 5 and the `benchmark.py` output
- the **chromosome 21** timing from cell 6
- `gpu_results.md` for your notes

If the speedup is lower than expected, say so and explain why: this kernel
launches once per chunk and scores one window per thread. Week 4's shared-memory
optimisation is where that number improves.